In [ ]:
from __future__ import print_function

import os

import keras
import tensorflow as tf
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Rescaling, BatchNormalization, GlobalAveragePooling2D
from keras.optimizers import RMSprop,Adam
import matplotlib.pyplot as plt
import numpy as np


batch_size = 12
num_classes = 3
epochs = 14
img_width = 128
img_height = 128
img_channels = 3
fit = True
_data_root = os.path.join(os.getcwd(), 'chest_xray')
train_dir = os.path.join(_data_root, 'train')
test_dir = os.path.join(_data_root, 'test')

with tf.device('/gpu:0'):
    
    #create training,validation and test datatsets
    train_ds,val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        train_dir,
        seed=123,
        validation_split=0.2,
        subset='both',
        image_size=(img_height, img_width),
        batch_size=batch_size,
        labels='inferred',
        shuffle=True)
    
    test_ds = tf.keras.preprocessing.image_dataset_from_directory(
        test_dir,
        seed=None,
        image_size=(img_height, img_width),
        batch_size=batch_size,
        labels='inferred',
        shuffle=True)

    class_names = train_ds.class_names
    print('Class Names: ',class_names)
    num_classes = len(class_names)

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.02),
        tf.keras.layers.RandomZoom(0.05),
        tf.keras.layers.RandomContrast(0.05),
    ])
    
    plt.figure(figsize=(10, 10))
    for images, labels in train_ds.take(2):
        for i in range(6):
            ax = plt.subplot(2, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(class_names[labels[i].numpy()])
            plt.axis("off")
    plt.savefig("run7_samples.png", bbox_inches="tight")
    plt.show(block=False)
    plt.pause(2)
    plt.close()

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(img_height, img_width, img_channels),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    #create model
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(img_height, img_width, img_channels)),
        data_augmentation,
        # Same as mobilenet preprocess
        Rescaling(scale=1.0 / 127.5, offset=-1.0),
        base_model,
        GlobalAveragePooling2D(), # reduces each feature map to a single value
        Dense(128, activation = 'relu'),
        Dropout(0.3),
        Dense(num_classes, activation = 'softmax')
    ])

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=Adam(learning_rate=0.0001),
                  metrics=['accuracy'])
    
    earlystop_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
    reduce_lr_callback = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    save_callback = tf.keras.callbacks.ModelCheckpoint("pneumonia.keras",save_freq='epoch',save_best_only=True)

    if fit:
        history = model.fit(
            train_ds,
            batch_size=batch_size,
            validation_data=val_ds,
            callbacks=[save_callback, earlystop_callback, reduce_lr_callback],
            epochs=epochs)
    else:
        try:
            model = tf.keras.models.load_model("pneumonia.keras")
        except Exception:
            try:
                model = tf.keras.models.load_model(
                    "pneumonia.keras",
                    custom_objects={
                        "preprocess_input": tf.keras.applications.mobilenet_v2.preprocess_input,
                    },
                )
            except Exception as e:
                raise RuntimeError("Error") from e

    #if shuffle=True when creating the dataset, samples will be chosen randomly   
    score = model.evaluate(test_ds, batch_size=batch_size)
    print('Test accuracy:', score[1])

    
    if fit:
        plt.figure()
        plt.plot(history.history['accuracy'])
        plt.plot(history.history['val_accuracy'])
        plt.title('model accuracy')
        plt.ylabel('accuracy')
        plt.xlabel('epoch')
        plt.legend(['train', 'val'], loc='upper left')
        plt.savefig("run7_accuracy.png", bbox_inches="tight")
        plt.show(block=False)
        plt.pause(2)
        plt.close()
        
    test_batch = test_ds.take(1)
    plt.figure(figsize=(10, 10))
    for images, labels in test_batch:
        for i in range(6):
            ax = plt.subplot(2, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            prediction = model.predict(tf.expand_dims(images[i].numpy(),0), verbose=0)#perform a prediction on this image
            plt.title('Actual:' + class_names[labels[i].numpy()]+ '\nPredicted:{} {:.2f}%'.format(class_names[np.argmax(prediction)], 100 * np.max(prediction)))
            plt.axis("off")
    plt.savefig("run7_predictions.png", bbox_inches="tight")
    plt.show(block=False)
    plt.pause(2)
    plt.close()

NameError: name '__file__' is not defined